# D_Gold - Modelo estrella

Gold consume las tablas Silver registradas en Unity Catalog y construye
4 dimensiones y 1 tabla de hechos, listas para Power BI.


In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

products = spark.table(f"{SILVER_SCHEMA}.products")


def write_gold(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


# Dimensión productos
df_dim_products = (
    products
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .drop("cocoa_percent")
)
write_gold(df_dim_products, f"{GOLD_SCHEMA}.dim_products")


print("GOLD - tabla dimensional din_products generado correctamente")
print("Dimensiones: dim_products")


In [0]:
%sql

SELECT 'dim_products' AS tabla, COUNT(*) AS total_filas FROM workspace.gold.dim_products



In [0]:
%sql
select * from workspace.gold.dim_products limit 2